In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option('display.width', 140)
np.set_printoptions(linewidth=130)

current_session = Path(locals()['__session__']).parent

df = pd.read_csv(current_session / 'train.csv')
tst_df = pd.read_csv(current_session / 'test.csv')

In [ ]:
modes = df.mode().iloc[0]
modes

In [ ]:
def proc_data(df_in):
    df = df_in.copy()
    df['Fare']= df.Fare.fillna(0)
    df.fillna(modes, inplace=True)
    df['LogFare'] = np.log1p(df['Fare'])
    df['Embarked'] = pd.Categorical(df.Embarked)
    df['Sex'] = pd.Categorical(df.Sex)
    return df

df = proc_data(df)
tst_df = proc_data(tst_df)

In [ ]:
cats = ['Embarked', 'Sex']
conts = ['Age', 'SibSp', 'Parch', 'LogFare', 'Pclass']
dep = 'Survived'

In [ ]:
df.Sex.head()

In [ ]:
df.Sex.cat.codes.head()


# Binary Splits

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig,axs = plt.subplots(1,2,figsize=(11,5))
sns.barplot(data=df, y=dep, x='Sex', ax=axs[0]).set(title='Survival Rate')
sns.countplot(data=df, x='Sex', ax=axs[1]).set(title='Histogram')

In [ ]:
from numpy import random
from sklearn.model_selection import train_test_split

random.seed(42)

trn_df, val_df = train_test_split(df, test_size=0.25)

In [ ]:
df[cats].apply(lambda x: x.cat.codes)

In [ ]:
trn_df[cats] = trn_df[cats].apply(lambda x: x.cat.codes)
val_df[cats] = val_df[cats].apply(lambda x: x.cat.codes)

In [ ]:
def xs_y(df):
    xs = df[cats+conts].copy()
    return xs, df[dep] if dep in df else None

trn_xs, trn_y = xs_y(trn_df)
val_xs, val_y = xs_y(val_df)

In [ ]:
preds = val_xs.Sex == 0

In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(val_y,preds)

In [ ]:
df_fare = trn_df[trn_df.LogFare > 0]

fig,axs = plt.subplots(1,2,figsize=(11,5))
sns.boxenplot(data=df_fare, x=dep, y='LogFare', ax=axs[0])#.set(title='Survival Rate')
sns.kdeplot(data=df_fare, x='LogFare', ax=axs[1])#.set(title='Histogram')

In [ ]:
preds = val_xs.LogFare > 2.7

In [ ]:
mean_absolute_error(val_y,preds)

In [ ]:
# standard deviation == how much are things similar within a group
def _side_score(side, y):
    tot = side.sum()
    if tot <= 1: return 0
    return y[side].std() * tot

In [ ]:
def score(col, y, split):
    lhs = col <= split
    return (_side_score(lhs, y) + _side_score(~lhs, y)) / len(y)

In [ ]:
score(trn_xs["Sex"], trn_y, 0.5)

In [ ]:
score(trn_xs["LogFare"], trn_y, 2.7)

In [ ]:
def iscore(nm, split):
    col = trn_xs[nm]
    return score(col, trn_y, split)

from ipywidgets import interact
interact(nm=conts, split=15.5)(iscore)

In [ ]:
interact(nm=cats, split=2.0)(iscore)

In [ ]:
nm="Age"
col = trn_xs[nm]
unq = col.unique()
unq.sort()
unq

In [ ]:
scores = np.array([score(col, trn_y,o) for o in unq if not np.isnan(o)])
unq[scores.argmin()]

In [ ]:
def min_col(df, nm):
    col,y = df[nm], df[dep]
    unq = col.dropna().unique()
    scores = np.array([score(col, y, o) for o in unq if not np.isnan(o)])
    idx = scores.argmin()
    return unq[idx], scores[idx]

min_col(trn_df, 'Age')

In [ ]:
cols = cats + conts

{o:min_col(trn_df, o) for o in cols}

In [ ]:
cols.remove('Sex') if 'Sex' in cols else None
is_male = trn_df.Sex == 1
males, females = trn_df.loc[is_male], trn_df.loc[~is_male]

In [ ]:
{o:min_col(males, o) for o in cols}

In [ ]:
{o:min_col(females, o) for o in cols}

We can see that the best binary split for females is `Pclass` <= 2 and `Age` <= 6 for males.

By adding these rules we have created a decision tree whether our model firstly check whether Sex is `male` or `female` and depending on the result it will check `Pclass` or `Age`.

To avoid manual coding of such tree; we can use `DecisionTreeClassifier`


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz

m = DecisionTreeClassifier(max_leaf_nodes=4).fit(trn_xs, trn_y)

In [ ]:
import graphviz
import re

def draw_tree(t, df, size=10, ratio=0.6, precision=2, **kwargs):
    s=export_graphviz(t, out_file=None,feature_names=df.columns, filled=True, rounded=True,
                      special_characters=True, rotate=False, precision=precision, **kwargs)
    return graphviz.Source(re.sub('Tree {', f'Tree {{ size={size}; ratio={ratio}', s))

In [ ]:
draw_tree(m, trn_xs, size=10)

In [ ]:
#gini

def gini(df, cond, dep):
    act = df.loc[cond, dep]
    print(act)
    print(act.mean())
    print(act.mean()**2)
    return 1 - act.mean()**2 - (1-act).mean()**2

In [ ]:
gini(df, df.Sex=='female',dep)

In [ ]:
mean_absolute_error(val_y, m.predict(val_xs))

In [ ]:
m = DecisionTreeClassifier(max_leaf_nodes=50)
m.fit(trn_xs, trn_y)

In [ ]:
draw_tree(m, trn_xs, size=20)

In [ ]:
mean_absolute_error(val_y, m.predict(val_xs))

In [ ]:
tst_df[cats] = tst_df[cats].apply(lambda x: x.cat.codes)
tst_xs, _ = xs_y(tst_df) 

In [ ]:
def subm(preds, suff):
    tst_df['Survived'] = preds
    sub_df = tst_df[['PassengerId', 'Survived']]
    sub_df.to_csv(f'sub-{suff}.csv',index=False)

subm(m.predict(tst_xs), 'tree')